# Disponibilidade dos produtos e desempenho comercial

**Técnico em Ciência de Dados — 2º ano**  
Análise Exploratória de Dados e Inteligência de Negócios  
U3 — Projeto AED | Semana 13 — Projeto AED – Desenvolvimento das Análises  
Aula 2: Investigação de padrões e relações nos dados

**Nome:** ____________________ **Turma:** __________

## Objetivos
- Investigar padrões, tendências e relações identificados nos dados do projeto.
- Confrontar volume, proporção e persistência da procura não atendida e relacionar as novas informações ao desempenho comercial.
- Investigar desequilíbrios de estoque entre lojas e registrar evidências e limites.

## Contexto e missão

Vocês integram a equipe de análise de dados de oito papelarias brasileiras. A espera e a disponibilidade geral não esclareceram suficientemente as diferenças de desempenho. Em resposta à solicitação da equipe, a direção forneceu registros de vendas, procura não atendida e estoques de seis produtos.

Pergunta investigativa: Quais condições estão associadas ao desempenho comercial das lojas e qual aspecto merece prioridade para uma primeira ação de melhoria?

Missão: Investiguem o que essas novas informações acrescentam à compreensão das diferenças de desempenho comercial e identifiquem oportunidades de utilizar melhor os estoques existentes.

Período: 06/10 a 30/11/2025. Dados simulados para fins didáticos; seis produtos do sortimento. Tempo: 43 minutos. Registrem Q1 a Q4 no notebook.

## Organização do ambiente e materiais

Organizem-se em duplas no laboratório de informática, com um computador por dupla. Acompanhem as orientações projetadas e mantenham o espaço de circulação do professor livre.

Alternem os papéis: um integrante executa as células e o outro confere as saídas e participa do registro das respostas.

- Roteiro da atividade.
- Notebook da atividade.
- `rede_papelarias_B2_integrado.csv`
- `rede_papelarias_produtos_S13A2.csv`
- 1 computador por dupla com Visual Studio Code, Jupyter, Python e pandas disponíveis; se utilizarem o Google Colab, acesso à internet.

No Google Colab, enviem os dois CSVs para a sessão antes de executar. Bibliotecas: pandas, NumPy (dependência de pandas) e IPython.display. O código está preparado; não é necessário alterá-lo.

## 1. Conheçam os dados complementares — 5 min

Reúnam o notebook e os dois CSVs na mesma pasta, selecionem o kernel Python e executem as células na ordem. Não são necessários arquivos produzidos em outras aulas.

Confiram os 2.304 registros de loja, produto e dia (384 loja-dias, 48 por loja). Observem vendas, reposições, estoques e unidades não atendidas. O notebook prepara e agrega os dados por loja-dia antes de integrá-los.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

arquivos = [Path("rede_papelarias_B2_integrado.csv"), Path("rede_papelarias_produtos_S13A2.csv")]
for arquivo in arquivos:
    if not arquivo.is_file():
        raise FileNotFoundError(f"Coloquem {arquivo.name} na mesma pasta do notebook.")
original = pd.read_csv(arquivos[0], encoding="utf-8-sig")
produtos = pd.read_csv(arquivos[1], encoding="utf-8-sig", parse_dates=["data"])
# Recompor a preparação da A1, preservando ausências.
lojas = original.drop_duplicates().copy()
assert lojas.id_registro.is_unique
for coluna in ["despesas_com_pessoal", "outras_despesas_operacionais"]:
    antes = lojas[coluna]
    texto = antes.astype("string").str.strip().str.replace("R$", "", regex=False).str.strip().str.replace(",", ".", regex=False)
    convertido = pd.to_numeric(texto, errors="raise")
    assert convertido.isna().sum() == antes.isna().sum()
    lojas[coluna] = convertido.astype(float)
lojas["data"] = pd.to_datetime(lojas.data)
lojas = lojas.loc[lojas.data.between("2025-10-06", "2025-11-30")].copy()
assert lojas.entradas.gt(0).all()
lojas["cupons_por_100_entradas"] = 100 * lojas.cupons_emitidos / lojas.entradas
lojas["resultado_apos_despesas"] = (lojas.lucro_bruto - lojas.investimento_divulgacao
    - lojas.despesas_com_pessoal - lojas.outras_despesas_operacionais)
print("Registros originais das lojas:", len(original))
print("Loja-dias no período:", len(lojas))
print("Registros dos produtos:", len(produtos))
print("Produtos distintos:", produtos.produto.nunique(), "| Lojas:", produtos.loja.nunique())
display(produtos.groupby("loja").data.nunique().rename("Dias de operação").to_frame())

### Dicionário dos dados complementares

| Campo | Unidade/tipo | Definição |
|---|---|---|
| id_registro_produto | texto | Identificador único do registro de produto. |
| id_registro_loja_dia | texto | Identificador do registro da loja e dia no CSV integrado da A1. |
| data | AAAA-MM-DD | Data da operação da loja. |
| loja | texto | Identificador da loja, L1 a L8. |
| produto | texto | Identificador do produto, P01 a P06. |
| descricao_produto | texto | Denominação comercial simulada. |
| estoque_inicial_un | inteiro, unidades | Estoque na abertura, antes de reposições do dia. |
| reposicao_un | inteiro, unidades | Quantidade recebida após a abertura e antes das vendas nesse modelo. Não inclui transferências entre lojas. |
| unidades_vendidas | inteiro, unidades | Unidades do produto vendidas no dia. Não equivale a número de cupons. |
| unidades_nao_atendidas | inteiro, unidades | Quantidade solicitada e não fornecida por indisponibilidade do produto (registro de solicitações, não clientes distintos nem vendas perdidas). |
| ruptura_no_dia | inteiro (0/1) | 1 se o estoque esteve zerado na abertura ou atingiu zero após as vendas; 0 caso contrário. Pode haver ruptura sem solicitação registrada não atendida. |
| estoque_final_un | inteiro, unidades | Saldo no fechamento do dia. |
| custo_unitario_aquisicao | R$/unidade | Custo simulado de aquisição unitária, constante por produto. |
| receita_liquida_produto | R$ | Receita do produto no dia, descontada a campanha simulada. |
| volume_unitario_l | litros/unidade | Volume externo aproximado por unidade embalada, assumido na simulação; não inclui corredores, prateleiras ou folgas. |

A disponibilidade geral refere-se à abertura da loja. Os seis produtos não representam todo o sortimento nem toda a procura. O resultado após despesas registradas não equivale a lucro líquido; cupons por 100 entradas não identifica pessoas distintas.

In [ ]:
# Conferir chaves, saldos e vínculo com cada loja-dia.
produtos = produtos.sort_values(["loja", "produto", "data"]).copy()
assert produtos.id_registro_produto.is_unique
assert not produtos.duplicated(["loja", "produto", "data"]).any()
assert not produtos.isna().any().any()
assert produtos.groupby(["loja", "data"]).size().eq(6).all()
campos_un = ["estoque_inicial_un", "reposicao_un", "unidades_vendidas", "unidades_nao_atendidas", "estoque_final_un"]
assert produtos[campos_un].ge(0).all().all()
assert all(pd.api.types.is_integer_dtype(produtos[c]) for c in campos_un)
assert (produtos.estoque_inicial_un + produtos.reposicao_un - produtos.unidades_vendidas == produtos.estoque_final_un).all()
anterior = produtos.groupby(["loja", "produto"]).estoque_final_un.shift()
assert produtos.loc[anterior.notna(), "estoque_inicial_un"].eq(anterior.dropna()).all()
assert produtos.ruptura_no_dia.eq((produtos.estoque_inicial_un.eq(0) | produtos.estoque_final_un.eq(0)).astype(int)).all()
assert (produtos.unidades_nao_atendidas.eq(0) | produtos.estoque_final_un.eq(0)).all()
vinculos = produtos.merge(lojas[["id_registro", "loja", "data"]], left_on=["id_registro_loja_dia", "loja", "data"], right_on=["id_registro", "loja", "data"], how="left", validate="many_to_one")
assert vinculos.id_registro.notna().all()
# Agregar os produtos ANTES de juntar aos indicadores comerciais.
produtos["procura"] = produtos.unidades_vendidas + produtos.unidades_nao_atendidas
produtos["semana"] = (produtos.data.sub(pd.Timestamp("2025-10-06")).dt.days // 7 + 1).astype(int)
produtos["cmv_sel"] = produtos.unidades_vendidas * produtos.custo_unitario_aquisicao
produtos["abertura_zero"] = produtos.estoque_inicial_un.eq(0).astype(int)
diario = produtos.groupby(["loja", "data"], as_index=False).agg(vendidas=("unidades_vendidas", "sum"), nao_atendidas=("unidades_nao_atendidas", "sum"), procura=("procura", "sum"), receita_sel=("receita_liquida_produto", "sum"), cmv_sel=("cmv_sel", "sum"), abertura_zero=("abertura_zero", "sum"))
integrado = lojas.merge(diario, on=["loja", "data"], how="left", validate="one_to_one")
assert len(integrado) == len(lojas) == len(diario)
assert integrado.procura.notna().all()
assert np.isclose(integrado.receita_liquida.sum(), lojas.receita_liquida.sum())
assert integrado.resultado_apos_despesas.isna().sum() == lojas.resultado_apos_despesas.isna().sum()
assert (integrado.receita_sel <= integrado.receita_liquida + .001).all()
assert (integrado.cmv_sel <= integrado.receita_liquida - integrado.lucro_bruto + .001).all()
assert (integrado.receita_sel - integrado.cmv_sel <= integrado.lucro_bruto + .001).all()
assert (integrado.receita_sel >= integrado.cmv_sel - .001).all()
assert integrado.loc[integrado.disponibilidade_produtos_pct.eq(100), "abertura_zero"].eq(0).all()
print("Conferências de chaves, saldos, integração e totais concluídas.")
print("Loja-dias após integração:", len(integrado))
colunas = {"loja":"Loja", "data":"Data", "descricao_produto":"Produto", "estoque_inicial_un":"Abertura (un.)", "reposicao_un":"Reposição (un.)", "unidades_vendidas":"Vendidas (un.)", "unidades_nao_atendidas":"Não atendidas (un.)", "estoque_final_un":"Fechamento (un.)"}
display(produtos.sort_values(["loja", "data", "produto"])[list(colunas)].head(6).rename(columns=colunas))

## 2. Investiguem os padrões de procura não atendida — 10 min

Calculem a proporção da procura registrada não atendida: [unidades não atendidas ÷ (unidades vendidas + unidades não atendidas)] × 100. Se não houver procura registrada, a proporção fica indefinida. Nos agrupamentos, utilizem a razão entre os totais, não a média dos percentuais.

Organizem os resultados por loja e produto e confiram a recorrência da procura não atendida ao longo das oito semanas.

In [ ]:
# Proporção pela razão entre totais; denominador zero permanece indefinido.
chave = ["loja", "produto"]
por_produto = produtos.groupby(chave).agg(descricao=("descricao_produto", "first"), procura=("procura", "sum"), vendas=("unidades_vendidas", "sum"), nao_atendidas=("unidades_nao_atendidas", "sum"), estoque_mediano=("estoque_final_un", "median"))
por_produto["pct_nao_atendida"] = 100 * por_produto.nao_atendidas / por_produto.procura.replace(0, np.nan)
semanal = produtos.groupby(chave + ["semana"]).agg(procura=("procura", "sum"), nao_atendidas=("unidades_nao_atendidas", "sum"), estoque_fim=("estoque_final_un", "last"))
por_produto["semanas_com_falta"] = semanal.nao_atendidas.gt(0).groupby(level=[0,1]).sum()
ordem = por_produto.sort_values("nao_atendidas", ascending=False)
print("Procura registrada:", produtos.procura.sum(), "unidades")
print("Vendidas:", produtos.unidades_vendidas.sum(), "| Não atendidas:", produtos.unidades_nao_atendidas.sum())
rotulos = {"descricao":"Produto", "procura":"Procura (un.)", "nao_atendidas":"Não atendida (un.)", "pct_nao_atendida":"Não atendida (%)", "semanas_com_falta":"Semanas com falta (de 8)"}
print("Oito primeiros casos em volume de procura não atendida:")
display(ordem[list(rotulos)].head(8).rename(columns=rotulos).round(2))
print("Unidades não atendidas por semana nas quatro primeiras combinações:")
display(semanal.nao_atendidas.unstack("semana").reindex(ordem.head(4).index).rename(columns=lambda s: f"Semana {s}"))

**Q1.** Identifiquem as duas combinações de loja e produto com maior volume de procura não atendida. Confrontem o volume, a proporção e a persistência dessas ocorrências. Comparem também a terceira e a quarta combinações em volume: a ordem permanece a mesma quando consideramos a proporção?

**Resposta:**

*Registrem aqui a resposta da dupla.*

## 3. Relacionem disponibilidade e desempenho comercial — 10 min

Comparem disponibilidade geral e proporção de procura não atendida com aproveitamento, receita líquida e resultado após despesas. Utilizem os casos completos da A1, nas oito semanas e na faixa de 150 a 199 entradas por dia.

Examinem as seis correlações de Pearson em cada recorte. Na tabela por loja, calculem a proporção não atendida pela razão entre os totais e confiram o número de observações.

In [ ]:
# Preservar o conjunto de casos completos da A1 e conferir o denominador adicional.
integrado["pct_nao_atendida"] = 100 * integrado.nao_atendidas / integrado.procura.replace(0, np.nan)
condicoes = ["disponibilidade_produtos_pct", "pct_nao_atendida"]
desempenho = ["cupons_por_100_entradas", "receita_liquida", "resultado_apos_despesas"]
variaveis_a1 = ["espera_media_min", "horas_equipe", "disponibilidade_produtos_pct"] + desempenho
completos_a1 = integrado.dropna(subset=variaveis_a1).copy()
completos = completos_a1.dropna(subset=["pct_nao_atendida"]).copy()
faixa = completos.loc[completos.entradas.between(150, 199)].copy()
print("Loja-dias no período:", len(integrado))
print("Casos completos da A1:", len(completos_a1))
print("Casos completos comuns:", len(completos))
print("Casos completos comuns na faixa de 150 a 199 entradas:", len(faixa))
rotulos_corr = {"disponibilidade_produtos_pct":"Disponibilidade geral", "pct_nao_atendida":"Procura não atendida (%)", "cupons_por_100_entradas":"Cupons por 100 entradas", "receita_liquida":"Receita líquida", "resultado_apos_despesas":"Resultado após despesas"}
def correlacoes(tab):
    return tab[condicoes + desempenho].corr(method="pearson").loc[condicoes, desempenho]
corr_periodo, corr_faixa = correlacoes(completos), correlacoes(faixa)
print("Pearson — período completo")
display(corr_periodo.rename(index=rotulos_corr, columns=rotulos_corr).round(3))
print("Pearson — faixa de movimento")
display(corr_faixa.rename(index=rotulos_corr, columns=rotulos_corr).round(3))

In [ ]:
# Comparar as oito lojas; proporção agregada pela razão dos totais.
def resumo_lojas(tab):
    res = tab.groupby("loja").agg(n=("id_registro", "size"), disponibilidade=("disponibilidade_produtos_pct", "mean"), espera=("espera_media_min", "mean"), aproveitamento=("cupons_por_100_entradas", "mean"), receita=("receita_liquida", "mean"), resultado=("resultado_apos_despesas", "mean"), procura=("procura", "sum"), nao_atendidas=("nao_atendidas", "sum"))
    res["pct_nao_atendida"] = 100 * res.nao_atendidas / res.procura.replace(0, np.nan)
    return res
lojas_periodo, lojas_faixa = resumo_lojas(completos), resumo_lojas(faixa)
rotulos = {"n":"Observações", "disponibilidade":"Disponibilidade geral (%)", "espera":"Espera média (min)", "nao_atendidas":"Não atendida (un.)", "procura":"Procura (un.)", "pct_nao_atendida":"Não atendida (%)", "aproveitamento":"Cupons por 100 entradas", "receita":"Receita média (R$)", "resultado":"Resultado médio (R$)"}
print("Oito lojas — faixa de 150 a 199 entradas")
display(lojas_faixa[list(rotulos)].rename(columns=rotulos).round(2))

As comparações usam registros no mesmo recorte de movimento, não necessariamente nas mesmas datas. Considerem as contagens. Associação não demonstra causalidade.

**Q2.** Na faixa de 150 a 199 entradas, o que a procura registrada não atendida acrescenta à comparação entre L3 e L4? Considerem também as demais lojas e as correlações nos dois recortes. As evidências permitem generalizar essa interpretação para toda a rede? Justifiquem.

**Resposta:**

*Registrem aqui a resposta da dupla.*

## 4. Investiguem desequilíbrios de estoque entre lojas — 12 min

Utilizem as oito semanas, sem restringir à faixa de movimento. Referência de triagem por loja e produto: (vendas totais em oito semanas ÷ 8) × 2, arredondando para cima.

Localizem as combinações cuja mediana do estoque final é superior à referência e cujo estoque no fechamento semanal ultrapassa essa referência em pelo menos seis das oito semanas. O critério é exploratório: não determina o estoque ideal.

Verifiquem as datas de coexistência: procura não atendida em uma loja e, em outra, estoque do mesmo produto acima da referência tanto na abertura quanto no fechamento, sem ruptura no dia.

Valor do estoque: estoque final em unidades × custo unitário de aquisição. Usem a data indicada, sem somar saldos de dias diferentes.

In [ ]:
# Triagem exploratória: referência de duas semanas de vendas médias.
por_produto["referencia_un"] = np.ceil(2 * por_produto.vendas / 8).astype(int)
semanal_ref = semanal.join(por_produto.referencia_un)
por_produto["semanas_acima"] = semanal_ref.estoque_fim.gt(semanal_ref.referencia_un).groupby(level=[0,1]).sum()
por_produto["candidato"] = por_produto.estoque_mediano.gt(por_produto.referencia_un) & por_produto.semanas_acima.ge(6)
candidatos = por_produto.loc[por_produto.candidato].copy()
rotulos = {"descricao":"Produto", "vendas":"Vendas em 8 semanas (un.)", "referencia_un":"Referência (un.)", "estoque_mediano":"Estoque final mediano (un.)", "semanas_acima":"Fechamentos acima da referência (de 8)"}
print("Combinações identificadas pelo critério de triagem:")
display(candidatos[list(rotulos)].rename(columns=rotulos))
print("Fechamentos semanais das combinações identificadas (unidades):")
display(semanal.estoque_fim.unstack("semana").reindex(candidatos.index).rename(columns=lambda s: f"Semana {s}"))

In [ ]:
# Procurar faltas e estoques do mesmo produto em outras lojas na mesma data.
com_ref = produtos.join(por_produto[["referencia_un", "candidato"]], on=["loja", "produto"])
falta = com_ref.loc[com_ref.unidades_nao_atendidas.gt(0), ["data", "produto", "loja", "unidades_nao_atendidas"]].rename(columns={"loja":"loja_falta"})
estoque = com_ref.loc[com_ref.candidato & com_ref.estoque_inicial_un.gt(com_ref.referencia_un) & com_ref.estoque_final_un.gt(com_ref.referencia_un) & com_ref.ruptura_no_dia.eq(0), ["data", "produto", "loja"]].rename(columns={"loja":"loja_estoque"})
coincidencias = falta.merge(estoque, on=["data", "produto"]).query("loja_falta != loja_estoque")
pares = coincidencias.groupby(["produto", "loja_falta", "loja_estoque"]).agg(dias_coincidentes=("data", "nunique"), unidades_nas_datas=("unidades_nao_atendidas", "sum")).sort_values("dias_coincidentes", ascending=False)
print("Contrastes em datas coincidentes:")
display(pares.rename(columns={"dias_coincidentes":"Dias coincidentes", "unidades_nas_datas":"Não atendidas nessas datas (un.)"}))
# Valor dos saldos dos dois contrastes mais recorrentes na data indicada.
chaves_saldo = pares.head(2).reset_index()[["loja_estoque", "produto"]].drop_duplicates().rename(columns={"loja_estoque":"loja"})
saldos = produtos.loc[produtos.data.eq(pd.Timestamp("2025-11-29"))].merge(chaves_saldo, on=["loja", "produto"], validate="one_to_one")
saldos["valor_estoque"] = saldos.estoque_final_un * saldos.custo_unitario_aquisicao
print("Saldos no fechamento de 29/11/2025 — dois contrastes mais recorrentes")
rotulos = {"loja":"Loja", "descricao_produto":"Produto", "estoque_final_un":"Estoque final (un.)", "custo_unitario_aquisicao":"Custo unitário (R$)", "valor_estoque":"Valor do estoque (R$)"}
display(saldos[list(rotulos)].rename(columns=rotulos).round(2))
print(f"Valor total dos saldos observados (R$): {saldos.valor_estoque.sum():.2f}")

O valor do estoque observado não é excesso comprovado nem economia realizável. A coexistência de falta e estoque em outra loja não significa que toda a procura possa ser atendida por transferência.

**Q3.** Identifiquem os dois contrastes mais recorrentes de procura não atendida e estoque persistentemente elevado do mesmo produto em outra loja, em datas coincidentes. Registrem as lojas, produtos, número de dias e evidências de persistência. Calculem o valor dos respectivos estoques no fechamento de 29/11/2025 e expliquem por que essas situações merecem investigação da direção.

**Resposta:**

*Registrem aqui a resposta da dupla.*

## 5. Consolidem as novas evidências — 6 min

**Q4.** Organizem três descobertas: uma sobre procura não atendida, outra sobre desempenho comercial e outra sobre distribuição de estoque. Para cada uma, registrem a evidência, a utilidade para a decisão da direção e um limite da interpretação.

| Descoberta | Evidência | Utilidade para a decisão da direção | Limite |
|---|---|---|---|
| Procura não atendida | | | |
| Desempenho comercial | | | |
| Distribuição de estoque | | | |

Preencham o quadro no notebook. Confiram os registros com o professor e salvem o notebook preenchido.

## Critérios para avaliação da atividade

- Confronta volume, proporção e persistência da procura não atendida com evidências.
- Integra os dados por produto aos indicadores de desempenho, respeitando os recortes.
- Identifica coincidências de procura não atendida e estoque elevado pelo critério de triagem.
- Quantifica estoques sem confundir seu valor com excesso comprovado ou economia.
- Registra descobertas, relevância e limites, sem afirmar causalidade ou prioridade definitiva.

Registro da atividade: notebook preenchido e salvo pela dupla. Sem envio ao AVA. A próxima aula não depende deste arquivo.